# 16 - Retrieval-Augmented Generation (RAG)


---

In the previous notebook, we learned about Modern Large Language Models (LLMs).

LLMs are powerful, but they have an important limitation:

They only know what they learned during training.

How can a model answer questions about:

- Your company documents?
- Yesterday's news?
- A PDF uploaded five minutes ago?

The answer is **Retrieval-Augmented Generation (RAG)**.

## 📜 History

Large Language Models can generate impressive answers.

However, they cannot automatically learn new information after training.

Imagine asking:

"What is inside my company's latest financial report?"

If the report was never part of training,

the model cannot know it.

Researchers solved this by retrieving relevant information before generating an answer.

This became known as **Retrieval-Augmented Generation (RAG)**.

## ❌ Why GPT Alone Was Not Enough

LLMs have several limitations:

- Knowledge cutoff
- Hallucinations
- Cannot access private documents
- Expensive to retrain

Instead of retraining,

what if we simply search for relevant information first?

That idea is RAG.

## 💡 Think Like a Researcher

Imagine you're building an AI assistant for a company.

Employees ask:

"What is our leave policy?"

The answer exists in a PDF.

Question:

Should you retrain GPT every time the PDF changes?

No.

Instead,

retrieve the relevant page,

then let the LLM answer from it.

## 💡 Core Idea

RAG has two stages:

1. Retrieval
2. Generation

```
Question

↓

Retrieve Relevant Documents

↓

Provide Context

↓

LLM Generates Answer
```

## 📊 Complete RAG Pipeline

```
Documents

↓

Chunking

↓

Embedding Model

↓

Vector Database

↓

User Question

↓

Question Embedding

↓

Similarity Search

↓

Top Chunks

↓

Prompt

↓

LLM

↓

Answer
```

##  Step 1 — Chunking

Large documents are divided into smaller pieces.

Example

PDF

↓

Page 1

↓

Paragraph 1

↓

Chunk

Chunking helps retrieval become faster and more accurate.

In [1]:
text = """
Artificial Intelligence is changing the world.

Machine Learning is a subset of AI.

Deep Learning uses neural networks.
"""

chunks = text.split("\n\n")

for i, chunk in enumerate(chunks, start=1):
    print(f"Chunk {i}")
    print(chunk)
    print("-" * 30)

Chunk 1

Artificial Intelligence is changing the world.
------------------------------
Chunk 2
Machine Learning is a subset of AI.
------------------------------
Chunk 3
Deep Learning uses neural networks.

------------------------------


##  Step 2 — Embeddings

Each chunk is converted into a vector.

Example

Chunk

↓

Embedding Model

↓

384-dimensional vector

Similar chunks have similar vectors.

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

chunks = [
    "Artificial Intelligence is changing the world.",
    "Machine Learning is a subset of AI.",
    "Deep Learning uses neural networks."
]

embeddings = model.encode(chunks)

print("Embedding shape:", embeddings.shape)

d:\Model_Evolution_Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2806.17it/s]


Embedding shape: (3, 384)


##  Step 3 — Vector Database

Embeddings are stored in a vector database.

Popular choices:

- Qdrant
- FAISS
- Pinecone
- Milvus
- Chroma

Unlike SQL databases,

vector databases search by semantic similarity.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

query = "What is Machine Learning?"

query_embedding = model.encode([query])

scores = cosine_similarity(query_embedding, embeddings)

print(scores)

[[0.43484896 0.74826205 0.5042592 ]]


In [6]:
import numpy as np

best_index = np.argmax(scores)

print("Retrieved Chunk:")
print(chunks[best_index])

Retrieved Chunk:
Machine Learning is a subset of AI.


##  Step 4 — Prompt Augmentation

The retrieved chunk is inserted into the prompt.

Example

Context:

Machine Learning is a subset of AI.

Question:

What is Machine Learning?

The LLM now answers using the retrieved context.

In [7]:
context = chunks[best_index]

question = "What is Machine Learning?"

prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

print(prompt)


Context:
Machine Learning is a subset of AI.

Question:
What is Machine Learning?

Answer:



##  Step 5 — Generation

The LLM reads:

- Retrieved Context
- User Question

and generates the final answer.

This reduces hallucinations because the model answers using retrieved evidence.

## ✅ Advantages

- Uses up-to-date information
- Works with private documents
- Reduces hallucinations
- No need to retrain the LLM
- Easy to update knowledge by changing documents

## ❌ Limitations

RAG also has challenges:

- Poor chunking can hurt retrieval.
- Weak embeddings reduce search quality.
- Bad retrieval leads to bad answers.
- Context window limits still apply.

## ✅ Summary

Today I learned:

- Why RAG was invented.
- The complete RAG pipeline.
- Chunking.
- Embeddings.
- Vector databases.
- Similarity search.
- Prompt augmentation.
- Generation.

## 🧠 Think Like a Researcher

Your RAG system can answer questions from documents.

Great!

But now users ask:

- Search the web.
- Send an email.
- Schedule a meeting.
- Query a database.
- Call an API.

A simple RAG pipeline cannot perform actions.

What if the AI could **plan**, **choose tools**, and **execute tasks**?

This is the next evolution:

**AI Agents.**